In [ ]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, ArrayType

spark = SparkSession.builder \
    .appName("MyDigitalTwin-Clustering") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Paths Docker vs local
if os.path.exists("/opt/spark/warehouse"):
    WAREHOUSE = "/opt/spark/warehouse"
else:
    # Windows local (chemin absolu)
    WAREHOUSE = os.path.abspath(os.path.join(os.getcwd(), "../../../scripts/notebooks", "..", "..", "warehouse"))
    if not os.path.exists(WAREHOUSE):
        WAREHOUSE = "C:/Users/arnau/Documents/MyDigitalTwin/warehouse"

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

---
## PARTIE B — Behavioral Clustering

On rassemble toutes les activités avec leurs features temporelles : heure, jour de la semaine, plateforme, poids d'interaction.

In [ ]:
# ── B1. CHARGEMENT DES FEATURES COMPORTEMENTALES ──────────────────────────────
# On utilise all_text déjà chargé (contient hour, weekday, platform, weight)
# + on ajoute tiktok_watch et instagram_likes (pas de texte utile mais données temporelles)

tiktok = read_table("tiktok_watch") \
    .select(F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"),
            F.lit("tiktok").alias("platform"),
            F.col("interaction_weight").alias("weight"),
            F.lit(None).cast(StringType()).alias("text"))

ig_likes = read_table("instagram_likes") \
    .select(F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"),
            F.lit("instagram").alias("platform"),
            F.col("interaction_weight").alias("weight"),
            F.lit(None).cast(StringType()).alias("text"))

# Union complète pour le behavioral
behavioral_raw = all_text.union(tiktok).union(ig_likes) \
    .select("hour", "weekday", "platform", "weight") \
    .filter(F.col("hour").isNotNull() & F.col("weekday").isNotNull())

print(f"Total events comportementaux : {behavioral_raw.count():,}")
behavioral_raw.groupBy("platform").count().orderBy(F.desc("count")).show()

In [ ]:
# ── B2. FEATURE ENGINEERING COMPORTEMENTAL ────────────────────────────────────
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

# Encoder la plateforme (catégorielle → numérique)
indexer = StringIndexer(inputCol="platform", outputCol="platform_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="platform_idx", outputCol="platform_ohe", dropLast=False)

# Normaliser l'heure (0-23 → cyclique via sin/cos pour que minuit soit proche de 23h)
behavioral_feat = behavioral_raw \
    .withColumn("hour_sin",   F.sin(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("hour_cos",   F.cos(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("weekday_sin", F.sin(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weekday_cos", F.cos(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weight_norm", F.coalesce(F.col("weight"), F.lit(1.0)).cast("double"))

# Fit indexer + encoder
indexer_model = indexer.fit(behavioral_feat)
behavioral_feat = indexer_model.transform(behavioral_feat)

encoder_model = encoder.fit(behavioral_feat)
behavioral_feat = encoder_model.transform(behavioral_feat)

# VectorAssembler : hour_sin/cos + weekday_sin/cos + weight + platform_ohe
assembler_beh = VectorAssembler(
    inputCols=["hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "weight_norm", "platform_ohe"],
    outputCol="beh_raw_features"
)

behavioral_feat = assembler_beh.transform(behavioral_feat)

# Standardisation
scaler_beh = StandardScaler(inputCol="beh_raw_features", outputCol="beh_features",
                             withMean=False, withStd=True)
scaler_model = scaler_beh.fit(behavioral_feat)
behavioral_feat = scaler_model.transform(behavioral_feat)

print("Features comportementales construites.")
behavioral_feat.select("hour", "weekday", "platform", "beh_features").show(5, truncate=80)

In [ ]:
# ── B3. KMEANS COMPORTEMENTAL (k=6) ───────────────────────────────────────────
K_BEHAVIORAL = 6

kmeans_beh = KMeans(
    featuresCol="beh_features",
    predictionCol="beh_cluster",
    k=K_BEHAVIORAL,
    seed=42,
    maxIter=50
)

print(f"Training K-Means behavioral (k={K_BEHAVIORAL})...")
km_beh_model = kmeans_beh.fit(behavioral_feat)
beh_df = km_beh_model.transform(behavioral_feat)

# Silhouette
evaluator_beh = ClusteringEvaluator(featuresCol="beh_features", predictionCol="beh_cluster")
sil_beh = evaluator_beh.evaluate(beh_df)
print(f"Silhouette Score (behavioral): {sil_beh:.4f}")

beh_df.groupBy("beh_cluster").count().orderBy("beh_cluster").show()

In [ ]:
# ── B4. CARACTERISATION DES CLUSTERS COMPORTEMENTAUX ─────────────────────────
# Pour chaque cluster : heure médiane, jour modal, plateforme dominante

beh_summary = beh_df.groupBy("beh_cluster").agg(
    F.round(F.avg("hour"), 1).alias("avg_hour"),
    F.round(F.avg("weekday"), 1).alias("avg_weekday"),
    F.count("*").alias("item_count")
).orderBy("beh_cluster")

beh_summary.show()

# Top plateforme par cluster
platform_labels = indexer_model.labels  # mapping index → platform name

beh_cluster_info = []
for cluster_id in range(K_BEHAVIORAL):
    subset = beh_df.filter(F.col("beh_cluster") == cluster_id)
    count  = subset.count()
    avg_h  = subset.agg(F.avg("hour")).collect()[0][0]
    avg_wd = subset.agg(F.avg("weekday")).collect()[0][0]
    
    top_plt = (
        subset.groupBy("platform").count()
        .orderBy(F.desc("count")).limit(3)
        .select("platform").rdd.flatMap(lambda x: x).collect()
    )
    
    # Heure humaine
    h = round(avg_h) if avg_h else 0
    if 5 <= h < 12:    period = "Matin"
    elif 12 <= h < 18: period = "Après-midi"
    elif 18 <= h < 23: period = "Soir"
    else:              period = "Nuit"
    
    wd = round(avg_wd) if avg_wd else 0
    day_type = "Weekend" if wd >= 5 else "Semaine"
    
    beh_cluster_info.append({
        "cluster_id": cluster_id,
        "item_count": count,
        "avg_hour": round(avg_h, 1) if avg_h else 0.0,
        "avg_weekday": round(avg_wd, 1) if avg_wd else 0.0,
        "time_period": period,
        "day_type": day_type,
        "top_platforms": top_plt
    })
    
    print(f"[Beh Cluster {cluster_id}] {count:,} items | {period} · {day_type} | Plateformes: {top_plt}")

In [ ]:
# ── B5. LABELLING DES CLUSTERS COMPORTEMENTAUX ────────────────────────────────
# À adapter après avoir regardé l'output ci-dessus !

BEH_LABELS = {
    0: {"label": "☀️ Journée active",      "emoji": "☀️"},
    1: {"label": "🌙 Nuit créative",        "emoji": "🌙"},
    2: {"label": "🎓 Mode étude",           "emoji": "🎓"},
    3: {"label": "🛋️ Loisirs soir",         "emoji": "🛋️"},
    4: {"label": "📅 Weekend détente",      "emoji": "📅"},
    5: {"label": "🛍️ Shopping & découverte","emoji": "🛍️"},
}

print("Labels définis. Mettre à jour BEH_LABELS après analyse ci-dessus.")

In [ ]:
# ── B6. ECRITURE behavioral_clusters ──────────────────────────────────────────
from pyspark.sql.types import DoubleType

beh_rows = []
for info in beh_cluster_info:
    cid = info["cluster_id"]
    beh_rows.append((
        cid,
        BEH_LABELS.get(cid, {}).get("label", f"Profil {cid}"),
        BEH_LABELS.get(cid, {}).get("emoji", "❓"),
        float(info["avg_hour"]),
        float(info["avg_weekday"]),
        info["time_period"],
        info["day_type"],
        info["top_platforms"],
        info["item_count"]
    ))

schema_beh = StructType([
    StructField("cluster_id",    IntegerType(), False),
    StructField("label",         StringType(),  False),
    StructField("emoji",         StringType(),  True),
    StructField("avg_hour",      DoubleType(),  True),
    StructField("avg_weekday",   DoubleType(),  True),
    StructField("time_period",   StringType(),  True),
    StructField("day_type",      StringType(),  True),
    StructField("top_platforms", ArrayType(StringType()), True),
    StructField("item_count",    LongType(),    True),
])

beh_clusters_df = spark.createDataFrame(beh_rows, schema_beh)

out_path_beh = os.path.join(WAREHOUSE, "behavioral_clusters")
beh_clusters_df.write.mode("overwrite").parquet(out_path_beh)

print(f"Ecrit dans : {out_path_beh}")
beh_clusters_df.show(truncate=50)

In [ ]:
spark.stop()
print("Spark session fermée. Notebook terminé.")